# Mercari Price Suggestion Challenge 


<img src='https://manexconsulting.com/wp-content/uploads/Price-Increase.jpg'>


Bu çalışmada `Mercari Price Suggestion Challenge` yarışması için ürün adı, kategori ve marka bilgileri kullanılarak ürün fiyat tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde fiyat tahmini ve veri hazırlama için gerekli kütüphaneleri içe aktarıyoruz.


In [2]:
!pip install py7zr -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.3/494.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 5.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import os
import zipfile
import tempfile
import py7zr

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [4]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [5]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/mercari-price-suggestion-challenge.zip'
print('Zip dosyası:', zip_path)
print('Zip mevcut mu?:', os.path.exists(zip_path))

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

print('Zip içindeki ilk dosyalar:', zip_members[:20])

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.tsv.7z')
test_member = find_zip_member(zip_members, 'test.tsv.7z')

print('Train member:', train_member)
print('Test member:', test_member)


Mounted at /content/drive
Zip dosyası: /content/drive/MyDrive/Colab Data Dosyaları/mercari-price-suggestion-challenge.zip
Zip mevcut mu?: True
Zip içindeki ilk dosyalar: ['sample_submission.csv.7z', 'sample_submission_stg2.csv.zip', 'test.tsv.7z', 'test_stg2.tsv.zip', 'train.tsv.7z']
Train member: train.tsv.7z
Test member: test.tsv.7z


## 3. Veri Dosyalarını Yükleme


In [6]:
# Bu bölümde .7z içindeki tsv dosyalarını çıkarıp örneklem kullanarak yüklüyoruz.


In [7]:
sample_train_rows = 300000
sample_test_rows = 100000


def read_7z_tsv_from_zip(zip_path, member_name, nrows=None):
    with tempfile.TemporaryDirectory() as tmpdir:
        local_7z_path = os.path.join(tmpdir, os.path.basename(member_name))

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            with zip_ref.open(member_name) as src_file, open(local_7z_path, 'wb') as dst_file:
                dst_file.write(src_file.read())

        with py7zr.SevenZipFile(local_7z_path, mode='r') as z:
            z.extractall(path=tmpdir)

        tsv_candidates = []
        for root, _, files in os.walk(tmpdir):
            for f in files:
                if f.endswith('.tsv'):
                    tsv_candidates.append(os.path.join(root, f))

        if not tsv_candidates:
            raise FileNotFoundError('7z içinden tsv çıkarılamadı.')

        tsv_path = tsv_candidates[0]
        return pd.read_csv(tsv_path, sep='	', nrows=nrows)

train = read_7z_tsv_from_zip(zip_path, train_member, nrows=sample_train_rows)
test = read_7z_tsv_from_zip(zip_path, test_member, nrows=sample_test_rows)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
train.head()


Train shape (sample): (300000, 8)
Test shape (sample): (100000, 7)


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & Parts,Razer,52.0,0,This keyboard is in great condition and works like it came out of the box. All of the ports are tested and work perfectly. The lights are customizable via the Razer Synapse app on your PC.
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,"Adorable top with a hint of lace and a key hole in the back! The pale pink is a 1X, and I also have a 3X available in white!"
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm] each. Stand about a foot high. They are being sold as a pair. Any questions please ask. Free shipping. Just got out of storage
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


## 4. Ön İşleme


In [8]:
# Bu bölümde eksik alanları düzenliyor ve metin/kategori bilgilerini sadeleştiriyoruz.


In [9]:
train_df = train.copy()
test_df = test.copy()

for df in [train_df, test_df]:
    df['category_name'] = df['category_name'].fillna('missing')
    df['brand_name'] = df['brand_name'].fillna('missing')
    df['item_description'] = df['item_description'].fillna('missing')
    df['name'] = df['name'].fillna('missing')

print('Train null count:', train_df.isnull().sum().sum())
print('Test null count:', test_df.isnull().sum().sum())
train_df.head()


Train null count: 0
Test null count: 0


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,missing,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & Parts,Razer,52.0,0,This keyboard is in great condition and works like it came out of the box. All of the ports are tested and work perfectly. The lights are customizable via the Razer Synapse app on your PC.
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,"Adorable top with a hint of lace and a key hole in the back! The pale pink is a 1X, and I also have a 3X available in white!"
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,missing,35.0,1,New with tags. Leather horses. Retail for [rm] each. Stand about a foot high. They are being sold as a pair. Any questions please ask. Free shipping. Just got out of storage
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,missing,44.0,0,Complete with certificate of authenticity


## 5. Özellik Çıkarımı


In [10]:
# Bu bölümde model girdilerini hazırlıyor ve kullanılacak alanları belirliyoruz.


In [11]:
feature_columns = [
    'name', 'item_condition_id', 'category_name', 'brand_name', 'shipping', 'item_description'
]

x = train_df[feature_columns].copy()
y = train_df['price'].copy()
test_x = test_df[feature_columns].copy()

for col in ['name', 'category_name', 'brand_name', 'item_description']:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42
)

categorical_features = ['name', 'category_name', 'brand_name', 'item_description']
numerical_features = ['item_condition_id', 'shipping']

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (240000, 6)
x_valid shape: (60000, 6)


## 6. Model Kurma


In [12]:
# Bu bölümde ürün bilgilerini kullanarak fiyat tahmini yapan başlangıç modelini kuruyoruz.


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['item_condition_id',
                                                   'shipping']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 min_frequency=10))]),
                                                  ['name', 'category_name',
                                                   'brand_name',
                                                   'item_description'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=18, min_samples_split=10,
                                       n_estimators=120, n_jobs=-1,
                                       random_state=42))])

## 7. RMSE Değerlendirmesi


In [14]:
# Bu bölümde modelin fiyat tahminindeki hata düzeyini RMSE ile ölçüyoruz.


In [15]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(y_valid, valid_preds)
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 32.90606


## 8. Test Tahmini ve Submission


In [16]:
# Bu bölümde test ürünleri için fiyat tahminlerini oluşturuyoruz.


In [17]:
test_preds = model.predict(test_x)

submission = pd.DataFrame({
    'test_id': test_df['test_id'].values,
    'price': test_preds
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (100000, 2)


,test_id,price
0,0,16.322124
1,1,16.612649
2,2,25.709462
3,3,21.703638
4,4,16.612649


In [18]:
output_path = '/content/mercari_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/mercari_submission.csv


## 9. Sonuç


Bu çalışmada Mercari Price Suggestion Challenge yarışması için ürün adı, marka, kategori ve açıklama bilgileri kullanılarak ürün fiyat tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 32.90606 RMSE değeri elde etti ve test verisi için fiyat tahminleri üretildi.